# Лабораторная работа № 2
## Базовая модель MNIST и проверка устойчивости входов
Цель: обучить воспроизводимый классификатор, проверить качество, валидацию входов и чувствительность к небольшим изменениям.

In [ ]:
from pathlib import Path
import json
import os
import sys
from IPython.display import display, Image

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'src' / 'lab2.py').exists():
    raise RuntimeError('Откройте Notebook из каталога проекта')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

def show_json(path):
    data = json.loads(Path(path).read_text(encoding='utf-8'))
    print(json.dumps(data, ensure_ascii=False, indent=2))
    return data

## 1. Проверка данных и обучение
Используется официальный тест MNIST. Точные пересечения с тестом, повторные обучающие изображения и противоречивые обучающие группы исключаются до стратифицированного разделения. Seed — 42.

Если модель уже обучена из терминала, Notebook использует сохранённый результат и не перезаписывает baseline.

In [ ]:
from src.lab2 import train_run
from src.model import load_model

if not Path('metrics/baseline.json').exists():
    train_run()
else:
    model = load_model()
    print('Загружена ранее обученная модель; SHA256 проверен.')

data_check = show_json('reports/lab2/data_check.json')

### Ограничения данных
MNIST достаточно разнообразен для учебного сравнения простого классификатора с константным ориентиром: представлены все десять классов и множество вариантов написания. Однако набор не представляет произвольные фотографии, цветные изображения и сложные фоны. В архиве отсутствуют идентификаторы авторов, а проверка точных дубликатов не выявляет все похожие изображения.

In [ ]:
training = show_json('metrics/training.json')

## 2. Качество модели
Основная метрика — macro-F1: каждый класс имеет одинаковый вес. Дополнительно рассчитываются Accuracy, macro-Precision, macro-Recall и метрики отдельных классов.

In [ ]:
baseline = show_json('metrics/baseline.json')

In [ ]:
display(Image(filename='reports/lab2/confusion_matrix.png'))
display(Image(filename='reports/lab2/errors.png'))
errors = show_json('reports/lab2/error_examples.json')

### Разбор ошибочных предсказаний
После просмотра рисунка опишите 2–3 конкретных примера: истинный класс, ответ модели и видимое сходство начертания. Линейный классификатор использует пиксели и ограниченно учитывает смещения и изменения формы штрихов.

## 3. Валидация и автоматические тесты
Допускаются массивы 28 × 28 и пакеты до 64 изображений. PNG ограничен 64 КиБ и 784 пикселями; поддерживается режим L. Некорректные значения отклоняются до вызова модели.

In [ ]:
print(Path('reports/lab2/pytest.txt').read_text(encoding='utf-8'))

## 4. Эксперимент с устойчивостью
Для 20 фиксированных тестовых изображений добавлен гауссовский шум с sigma=8. Предполагается, что небольшое изменение сохраняет истинную цифру; это проверяется также визуально.

In [ ]:
robustness = show_json('reports/lab2/robustness.json')
display(Image(filename='reports/lab2/robustness.png'))
display(Image(filename='reports/lab2/unusual_input.png'))
print('Доля изменившихся ответов:', robustness['changed_answer_fraction'])
print('Accuracy до:', robustness['accuracy_before'])
print('Accuracy после:', robustness['accuracy_after'])

## 5. Вывод
Построена линейная базовая модель, сохранены параметры и метрики. Валидация ограничивает структуру и размер входов. Случайный шум показывает, что формально допустимый вход не обязательно содержит цифру. Дальнейшая проверка — обучение с шумовыми искажениями и оценка механизма отказа на отдельной выборке нецифровых изображений. Эксперимент на 20 объектах не доказывает общую устойчивость модели.